# Neo4j Semantic RAG Baseline

Семантический RAG baseline на тех же форумных текстах, что используются в проекте GraphRAG.

Ноутбук делает следующее:
1. Загружает 5 JSON-файлов из `messages/`.
2. Преобразует посты в `LlamaIndex Document` с metadata.
3. Делит тексты на чанки.
4. Строит векторный индекс.
5. Выполняет semantic retrieval по вопросу.
6. Генерирует ответ на естественном языке на основе найденных чанков.

Это baseline для сравнения с ontology-driven GraphRAG.


In [41]:
# 0) Environment / config
import os
from pathlib import Path

BASE_DIR = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф')
MESSAGES_DIR = BASE_DIR / 'messages'
PERSIST_DIR = BASE_DIR / 'retrieval_outputs' / 'llamaindex_semantic_rag'
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or os.getenv('GPT_API_KEY')
EMBED_MODEL = os.getenv('SEM_RAG_EMBED_MODEL', 'text-embedding-3-large')
LLM_MODEL = os.getenv('SEM_RAG_LLM_MODEL', 'gpt-5-mini')

print('BASE_DIR =', BASE_DIR)
print('MESSAGES_DIR =', MESSAGES_DIR)
print('PERSIST_DIR =', PERSIST_DIR)
print('OPENAI_API_KEY / GPT_API_KEY set =', bool(OPENAI_API_KEY))
print('EMBED_MODEL =', EMBED_MODEL)
print('LLM_MODEL =', LLM_MODEL)

if not OPENAI_API_KEY:
    raise ValueError('Neither OPENAI_API_KEY nor GPT_API_KEY is set.')


BASE_DIR = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф
MESSAGES_DIR = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/messages
PERSIST_DIR = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/llamaindex_semantic_rag
OPENAI_API_KEY / GPT_API_KEY set = True
EMBED_MODEL = text-embedding-3-large
LLM_MODEL = gpt-5-mini


In [42]:
# 1) Install dependencies if needed
# Если пакеты уже стоят, ячейку можно пропустить.
%pip install -q llama-index llama-index-embeddings-openai llama-index-llms-openai pandas tiktoken requests


Note: you may need to restart the kernel to use updated packages.


In [43]:
# 2) Imports
import json
import math
import re
import csv
import requests
from collections import Counter

import pandas as pd
from pathlib import Path
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI


## 3. Загрузка корпуса

Корпус состоит из 5 JSON-файлов:
- `messages_fr_1.json`
- `messages_fr_2.json`
- `messages_germany.json`
- `messages_italy.json`
- `messages_spain.json`

Каждый объект содержит:
- `post_id`
- `post_url`
- `author`
- `date`
- `text`


In [44]:
# 3) Load source JSON files
MESSAGE_FILES = [
    'messages_fr_1.json',
    'messages_fr_2.json',
    'messages_germany.json',
    'messages_italy.json',
    'messages_spain.json',
]

SOURCE_TO_COUNTRY = {
    'messages_fr_1.json': 'france',
    'messages_fr_2.json': 'france',
    'messages_germany.json': 'germany',
    'messages_italy.json': 'italy',
    'messages_spain.json': 'spain',
}

rows = []
for filename in MESSAGE_FILES:
    path = MESSAGES_DIR / filename
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    for item in data:
        rows.append({
            'source_file': filename,
            'country_source': SOURCE_TO_COUNTRY[filename],
            'post_id': item.get('post_id'),
            'post_url': item.get('post_url'),
            'author': item.get('author'),
            'date': item.get('date'),
            'text': (item.get('text') or '').strip(),
        })

df = pd.DataFrame(rows)
df['text_len_chars'] = df['text'].str.len()
df['text_len_words'] = df['text'].str.split().str.len()

print('posts =', len(df))
print(df.groupby('country_source').size())
df.head(3)


posts = 436
country_source
france     116
germany    210
italy       87
spain       23
dtype: int64


,source_file,country_source,post_id,post_url,author,date,text,text_len_chars,text_len_words
0,messages_fr_1.json,france,3902451,https://forum.awd.ru/viewtopic.php?p=3902451#p...,!Casper!,"11 июн 2013, 09:05","Прилетали вдвоем с другом в Париж, ШДГ, друг п...",572,88
1,messages_fr_1.json,france,10918979,https://forum.awd.ru/viewtopic.php?p=10918979#...,Melkaya13,"12 май 2022, 22:54",Летала в апреле во Францию по однократной эсто...,353,55
2,messages_fr_1.json,france,12265196,https://forum.awd.ru/viewtopic.php?p=12265196#...,constsddlem,"01 янв 2026, 21:27",За год 5 раз проходил паспортный контроль в CD...,296,54


In [45]:
# 4) Quick corpus statistics
stats = df.groupby('country_source').agg(
    posts=('post_id', 'count'),
    avg_chars=('text_len_chars', 'mean'),
    avg_words=('text_len_words', 'mean'),
    min_words=('text_len_words', 'min'),
    max_words=('text_len_words', 'max'),
).round(1)
stats


,posts,avg_chars,avg_words,min_words,max_words
country_source,,,,,
france,116,351.1,54.7,8,231
germany,210,519.5,78.4,11,300
italy,87,341.2,54.1,5,201
spain,23,293.5,44.9,8,92


## 4. Преобразование постов в `LlamaIndex Document`

Каждый пост становится отдельным документом.

В metadata сохраняются:
- источник (`source_file`)
- страна корпуса (`country_source`)
- `post_id`
- `post_url`
- `author`
- `date`

Это важно, чтобы потом можно было:
- делать фильтрацию по стране,
- показывать источник ответа,
- анализировать retrieval.


In [46]:
# 5) Convert posts to LlamaIndex Documents

def row_to_document(row: pd.Series) -> Document:
    return Document(
        text=row['text'],
        metadata={
            'source_file': row['source_file'],
            'country_source': row['country_source'],
            'post_id': str(row['post_id']),
            'post_url': row['post_url'],
            'author': row['author'],
            'date': row['date'],
        },
    )

documents = [row_to_document(row) for _, row in df.iterrows()]
print('documents =', len(documents))
print(documents[0].metadata)
print(documents[0].text[:400])


documents = 436
{'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '3902451', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=3902451#p3902451', 'author': '!Casper!', 'date': '11 июн 2013, 09:05'}
Прилетали вдвоем с другом в Париж, ШДГ, друг прошел контроль за 10 сек максиму без каких либо вопросов, а вот мне устроили допрос минут на 5. Попросили показать обратный билет, показать страховку, бронь отеля, спросили оплачен ли отель, показать кредитные карты, наличные деньги, спросили сколько наличности при себе, спросили какой картой я оплачивал отель, цель визита, на сколько дней приехал, кем


## 5. Chunking

Для semantic RAG индексировать целые посты не всегда оптимально.
Поэтому тексты делятся на чанки.

Здесь используется `SentenceSplitter` с параметрами:
- `chunk_size = 700`
- `chunk_overlap = 120`

Это разумный baseline для форумных текстов средней длины.


In [47]:
# 6) Chunking settings
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)
print('nodes =', len(nodes))
print('avg nodes per post =', round(len(nodes) / len(documents), 2))
print('\nSample node text:\n')
print(nodes[0].text[:500])
print('\nSample node metadata:\n')
print(nodes[0].metadata)


nodes = 449
avg nodes per post = 1.03

Sample node text:

Прилетали вдвоем с другом в Париж, ШДГ, друг прошел контроль за 10 сек максиму без каких либо вопросов, а вот мне устроили допрос минут на 5. Попросили показать обратный билет, показать страховку, бронь отеля, спросили оплачен ли отель, показать кредитные карты, наличные деньги, спросили сколько наличности при себе, спросили какой картой я оплачивал отель, цель визита, на сколько дней приехал, кем я работаю в России. После этого поставили штамп в паспорт! Вывод: держите все распечатки билетов, б

Sample node metadata:

{'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '3902451', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=3902451#p3902451', 'author': '!Casper!', 'date': '11 июн 2013, 09:05'}


## 6. Настройка моделей

Используются:
- OpenAI embeddings для dense retrieval
- OpenAI LLM для answer-stage

Если нужен только retrieval baseline, answer-stage можно не использовать.


In [48]:
# 7) Configure LlamaIndex models
from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
GPT_API_KEY = os.getenv("GPT_API_KEY")

if not GPT_API_KEY:
    raise ValueError('GPT_API_KEY is not set in environment.')

Settings.embed_model = OpenAIEmbedding(
    model=EMBED_MODEL,
    api_key=GPT_API_KEY,
)
Settings.llm = OpenAI(
    model=LLM_MODEL,
    api_key=GPT_API_KEY,
)

print('Embedding model configured:', EMBED_MODEL)
print('LLM configured:', LLM_MODEL)


Embedding model configured: text-embedding-3-large
LLM configured: gpt-5-mini


## 7. Построение и сохранение индекса

Индекс строится один раз и сохраняется на диск.
При повторном запуске его можно просто загрузить.


In [49]:
# 8) Build index and persist it
REBUILD_INDEX = False

if REBUILD_INDEX or not (PERSIST_DIR / 'docstore.json').exists():
    index = VectorStoreIndex(nodes)
    index.storage_context.persist(persist_dir=str(PERSIST_DIR))
    print('Index built and persisted to', PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=str(PERSIST_DIR))
    index = load_index_from_storage(storage_context)
    print('Index loaded from', PERSIST_DIR)


Index loaded from /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/llamaindex_semantic_rag


## 8. Семантический retrieval

На этом этапе выполняется обычный semantic search:
- вопрос преобразуется в embedding,
- из индекса извлекаются top-k наиболее близких чанков.


In [50]:
# 9) Simple retrieval helper

def retrieve_chunks(question: str, top_k: int = 5):
    retriever = VectorIndexRetriever(
        index=index,
        similarity_top_k=top_k,
    )
    return retriever.retrieve(question)


In [51]:
# 10) Example retrieval
QUESTION = 'Which documents do I need to bring to passport control in Paris?'
retrieved_nodes = retrieve_chunks(QUESTION, top_k=5)

for i, node in enumerate(retrieved_nodes, 1):
    print('=' * 100)
    print(f'Rank #{i}')
    print('score =', getattr(node, 'score', None))
    print('metadata =', node.metadata)
    print(node.text[:900])
    print()


Rank #1
score = 0.5328646420321396
metadata = {'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '11484564', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=11484564#p11484564', 'author': 'r41nyday', 'date': '28 май 2023, 10:16'}
Проходила паспортный контроль в третьем терминале CDG пару недель назад. Очередь была приличная, я была ближе к началу и ждала минут 20. Всех очень быстро пропускали, меня спросили только шенгенская ли у меня виза (был неоткрытый французский шенген). Всё - ни отпечатки, ни брони, ни обратный билет не просили. Всем удачи!

Rank #2
score = 0.5298865332004468
metadata = {'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '11769523', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=11769523#p11769523', 'author': 'nov_n', 'date': '26 мар 2024, 12:24'}
29 февраля, 11-30 - 12-00, Шарль де Голь, Терминал 1... для международного аэропорта на паспортном контроле очередь была небольшая, но двигалась медленно. Судя по 

## 9. Генерация ответа

Ниже строится answer-stage поверх найденных чанков.
Это уже полноценный text-based semantic RAG baseline.


In [52]:
# 11) Query engine for answer generation
retriever = VectorIndexRetriever(index=index, similarity_top_k=5)
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.0)],
)


In [53]:
# 12) Ask a question
QUESTION = 'Which documents do I need to bring to passport control in Paris?'
response = query_engine.query(QUESTION)
print(str(response))


Bring everything that can quickly confirm your identity, purpose and means of stay. Useful documents:

- Passport with a valid Schengen visa (if required).  
- Boarding pass and return/ onward ticket (printed or on phone).  
- Hotel reservation/booking confirmation (printed); if possible, proof that it’s paid.  
- Travel insurance certificate.  
- Proof of funds: cash, bank/credit cards (or recent bank statement).  
- Printed copies of flight tickets, bookings and your travel itinerary.  
- If relevant, a letter from your employer or other documents confirming your purpose of visit.

Keep these documents handy and organized — officers often ask about length of stay, where you will stay and whether you have a return ticket.


## 10. Посмотреть grounding ответа

Эта ячейка показывает, на каких чанках был основан ответ.


In [54]:
# 13) Inspect source nodes used by answer stage
for i, src in enumerate(response.source_nodes, 1):
    print('=' * 100)
    print(f'Source #{i}')
    print('score =', src.score)
    print('metadata =', src.node.metadata)
    print(src.node.text[:1000])
    print()


Source #1
score = 0.5328557187718621
metadata = {'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '11484564', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=11484564#p11484564', 'author': 'r41nyday', 'date': '28 май 2023, 10:16'}
Проходила паспортный контроль в третьем терминале CDG пару недель назад. Очередь была приличная, я была ближе к началу и ждала минут 20. Всех очень быстро пропускали, меня спросили только шенгенская ли у меня виза (был неоткрытый французский шенген). Всё - ни отпечатки, ни брони, ни обратный билет не просили. Всем удачи!

Source #2
score = 0.529900192025092
metadata = {'source_file': 'messages_fr_1.json', 'country_source': 'france', 'post_id': '11769523', 'post_url': 'https://forum.awd.ru/viewtopic.php?p=11769523#p11769523', 'author': 'nov_n', 'date': '26 мар 2024, 12:24'}
29 февраля, 11-30 - 12-00, Шарль де Голь, Терминал 1... для международного аэропорта на паспортном контроле очередь была небольшая, но двигалась медленно. Судя 

## 11. Удобная функция для экспериментов


In [55]:
# 14) Helper: retrieve + answer + sources

def sem_rag_answer(question: str, top_k: int = 5):
    retriever = VectorIndexRetriever(index=index, similarity_top_k=top_k)
    query_engine = RetrieverQueryEngine.from_args(retriever=retriever)
    response = query_engine.query(question)
    result = {
        'question': question,
        'answer': str(response),
        'sources': [],
    }
    for src in response.source_nodes:
        result['sources'].append({
            'score': src.score,
            'metadata': src.node.metadata,
            'text': src.node.text,
        })
    return result


In [56]:
# 15) Example experiment questions
TEST_QUESTIONS = [
    'Which documents do I need to bring to passport control in Paris?',
    'What questions are most often asked at passport control in France?',
    'How often are fingerprints taken when entering France?',
    'Which documents are checked in Paris but not in Nice?',
]

for q in TEST_QUESTIONS:
    print('\n' + '#' * 120)
    print('QUESTION:', q)
    res = sem_rag_answer(q, top_k=5)
    print('ANSWER:', res['answer'])



########################################################################################################################
QUESTION: Which documents do I need to bring to passport control in Paris?
ANSWER: Take originals and printed copies, and keep them handy for inspection:

- Passport with your visa (Schengen visa if required)  
- Boarding passes (arrival and return) / proof of onward ticket  
- Hotel reservation(s) — printed confirmation; proof of payment if available  
- Travel insurance certificate  
- Proof of funds: cash and / or bank cards (and screenshots/statements if you like)  
- Supporting docs if asked: e.g., employment details or invitation letter

Tip: have all documents printed and ready to show — checks are often quick but sometimes officers ask for several confirmations.

########################################################################################################################
QUESTION: What questions are most often asked at passport control in France?


## 12. Комментарий для сравнения с GraphRAG

Этот ноутбук реализует **semantic text RAG baseline**.

Его сильные стороны:
- быстрое построение,
- хорошая работа на описательных и fuzzy-вопросах,
- отсутствие необходимости в онтологии.

Его слабые стороны относительно твоего GraphRAG:
- агрегаты считаются нестрого,
- conditional / difference / intersect вопросы решаются хуже,
- multihop-структура не задана явно,
- answer-stage сильнее зависит от LLM-интерпретации retrieved chunks.

Именно поэтому этот baseline полезно сравнивать с ontology-driven GraphRAG на одном и том же наборе вопросов.


## 13. Evaluation: Semantic RAG vs GraphRAG

Ниже добавляется evaluation-слой для сравнения двух retrieval-подходов на одном и том же gold:

- **Semantic RAG**: question -> retrieved chunks -> LLM extraction -> canonical triples / aggregate result
- **GraphRAG**: используется уже посчитанный baseline из `summary_qnums.csv`

### Важная оговорка по метрикам

Raw graph triples в `annotaion_q_graph` содержат внутренние ключи графа (`Encounter`, `DocumentCheck`, `DocumentInstance` и т.д.), которые нельзя напрямую восстановить из исходных текстов semantic RAG-бейзлайном. Поэтому для semantic comparison используется **canonical encounter-level representation**:

- `Encounter(post_id) -atCountry-> Country`
- `Encounter(post_id) -atCity-> City`
- `Encounter(post_id) -atAirport-> Airport`
- `Encounter(post_id) -documentType-> Literal`
- `Encounter(post_id) -topicKey-> Literal`
- `Encounter(post_id) -biometricModality-> Literal`

Gold-аннотации приводятся к этой же canonical форме, после чего считаются `precision / recall / f1`.

Для aggregate-вопросов semantic RAG извлекает encounter-level facts, затем Python считает aggregate result по gold-метаданным (`group_by`, `metric`, `order`, `top_k`) и сравнивает его с `aggregate_output`.


In [57]:
# 16) Evaluation config
ANNOT_QG_DIR = BASE_DIR / 'annotaion_q_graph'
GRAPH_RESULTS_SOURCE = BASE_DIR / 'summary_qnums.csv'
QNUMS = [1, 9, 10]
SEM_TOP_K = 5
EXTRACT_LLM_MODEL = os.getenv('SEM_RAG_EXTRACT_MODEL', LLM_MODEL)
COMPARE_WITH_GRAPH = True

META_KEYS = ['intents','mode','conditional','scope','exclude_scope','doc_types','topic_keys','biometric_modalities','aggregate']
COUNTRY_SOURCE_TO_KEY = {
    'france': 'country_france',
    'germany': 'country_germany',
    'italy': 'country_italy',
    'spain': 'country_spain',
}

print('ANNOT_QG_DIR =', ANNOT_QG_DIR)
print('GRAPH_RESULTS_SOURCE =', GRAPH_RESULTS_SOURCE)
print('QNUMS =', QNUMS)
print('SEM_TOP_K =', SEM_TOP_K)
print('EXTRACT_LLM_MODEL =', EXTRACT_LLM_MODEL)
print('COMPARE_WITH_GRAPH =', COMPARE_WITH_GRAPH)


ANNOT_QG_DIR = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotaion_q_graph
GRAPH_RESULTS_SOURCE = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/summary_qnums.csv
QNUMS = [1, 9, 10]
SEM_TOP_K = 5
EXTRACT_LLM_MODEL = gpt-5-mini
COMPARE_WITH_GRAPH = True


In [58]:
# 17) Gold loading and canonicalization helpers

def annotation_files_for_qnum(qnum: int):
    qdir = ANNOT_QG_DIR / f'q_{qnum}'
    if not qdir.exists():
        raise FileNotFoundError(f'Question directory not found: {qdir}')
    files = sorted(qdir.glob('*/*.json'))
    ann = [f for f in files if f.name.startswith('annotation_') and f.suffix == '.json']
    return ann


def get_question_text(qnum: int) -> str:
    files = annotation_files_for_qnum(qnum)
    if not files:
        raise ValueError(f'No annotation files found for q_{qnum}')
    data = json.loads(files[0].read_text(encoding='utf-8'))
    question = data.get('question')
    if not question:
        raise ValueError(f'Question text not found in {files[0]}')
    return question


def triples_to_set(rows):
    return {
        (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
        for r in rows
        if all(k in r for k in ['s_label','s_key','p','o_label','o_key'])
    }


def load_gold(qnum: int):
    qdir = ANNOT_QG_DIR / f'q_{qnum}'
    triples = []
    metas = []
    for f in annotation_files_for_qnum(qnum):
        data = json.loads(f.read_text(encoding='utf-8'))
        for t in data.get('triples', []):
            s = t.get('s', {})
            o = t.get('o', {})
            triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
        meta = {k: data.get(k) for k in META_KEYS if k in data}
        if meta:
            metas.append(meta)
    meta_ref = metas[0] if metas else {}
    meta_consistent = all(m == meta_ref for m in metas) if metas else True

    agg_ref = None
    agg_path = qdir / 'aggregate_output.json'
    if agg_path.exists():
        agg_ref = json.loads(agg_path.read_text(encoding='utf-8'))

    return set(triples), meta_ref, meta_consistent, agg_ref


def norm_agg(rows):
    if rows is None:
        return rows
    out = []
    for r in rows:
        if isinstance(r, dict):
            out.append({k: r.get(k) for k in sorted(r.keys())})
        else:
            out.append(r)
    if out and isinstance(out[0], dict) and 'metric' in out[0]:
        key_fields = [k for k in out[0].keys() if k != 'metric']
        key_field = key_fields[0] if key_fields else None
        def sort_key(d):
            try:
                m_val = float(d.get('metric'))
            except Exception:
                m_val = 0.0
            k = d.get(key_field) if key_field else ''
            return (-m_val, k)
        out = sorted(out, key=sort_key)
    return out


def norm_meta(v):
    if v == '':
        return None
    if isinstance(v, list):
        return sorted(v)
    if isinstance(v, dict):
        return {kk: sorted(vv) if isinstance(vv, list) else vv for kk, vv in v.items()}
    return v


def _extract_post_id_from_key(value: str):
    if not value or not isinstance(value, str):
        return None
    m = re.search(r'encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'questioning_encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'doccheck_encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'documentinstance_encounter_(\d+)', value)
    if m:
        return m.group(1)
    return None


def _canonical_post_key_from_annotation(data: dict):
    for t in data.get('triples', []):
        for side in ['s', 'o']:
            key = (t.get(side) or {}).get('key')
            post_id = _extract_post_id_from_key(key)
            if post_id:
                return f'post_{post_id}'
    return None


def load_gold_canonical(qnum: int):
    canonical = set()
    encounters = {}
    meta_ref = None
    _, meta_ref, _, agg_ref = load_gold(qnum)

    for f in annotation_files_for_qnum(qnum):
        data = json.loads(f.read_text(encoding='utf-8'))
        post_key = _canonical_post_key_from_annotation(data)
        if not post_key:
            continue
        enc = encounters.setdefault(post_key, {
            'post_key': post_key,
            'atCountry': set(),
            'atCity': set(),
            'atAirport': set(),
            'document_types': set(),
            'topic_keys': set(),
            'biometric_modalities': set(),
        })
        for t in data.get('triples', []):
            s = t.get('s', {})
            o = t.get('o', {})
            p = t.get('p')
            if s.get('label') == 'Encounter' and p in {'atCountry', 'atCity', 'atAirport'}:
                canonical.add((s.get('label'), post_key, p, o.get('label'), o.get('key')))
                if p == 'atCountry':
                    enc['atCountry'].add(o.get('key'))
                elif p == 'atCity':
                    enc['atCity'].add(o.get('key'))
                elif p == 'atAirport':
                    enc['atAirport'].add(o.get('key'))
            elif p == 'documentType' and o.get('key'):
                canonical.add(('Encounter', post_key, 'documentType', 'Literal', o.get('key')))
                enc['document_types'].add(o.get('key'))
            elif p == 'topicKey' and o.get('key'):
                canonical.add(('Encounter', post_key, 'topicKey', 'Literal', o.get('key')))
                enc['topic_keys'].add(o.get('key'))
            elif p == 'biometricModality' and o.get('key'):
                canonical.add(('Encounter', post_key, 'biometricModality', 'Literal', o.get('key')))
                enc['biometric_modalities'].add(o.get('key'))

    return canonical, encounters, meta_ref, agg_ref


In [59]:
# 18) Candidate keys and retrieval by qnum

def load_candidate_keys_from_gold():
    cities, airports, countries = set(), set(), set()
    doc_types, topic_keys, biometric_modalities = set(), set(), set()

    for qdir in sorted(ANNOT_QG_DIR.glob('q_*')):
        for f in sorted(qdir.glob('*/*.json')):
            if not f.name.startswith('annotation_'):
                continue
            data = json.loads(f.read_text(encoding='utf-8'))
            for t in data.get('triples', []):
                s = t.get('s', {})
                o = t.get('o', {})
                p = t.get('p')
                if o.get('label') == 'City':
                    cities.add(o.get('key'))
                if o.get('label') == 'Airport':
                    airports.add(o.get('key'))
                if o.get('label') == 'Country':
                    countries.add(o.get('key'))
                if p == 'documentType' and o.get('key'):
                    doc_types.add(o.get('key'))
                if p == 'topicKey' and o.get('key'):
                    topic_keys.add(o.get('key'))
                if p == 'biometricModality' and o.get('key'):
                    biometric_modalities.add(o.get('key'))

    return {
        'cities': sorted(x for x in cities if x),
        'airports': sorted(x for x in airports if x),
        'countries': sorted(x for x in countries if x),
        'doc_types': sorted(x for x in doc_types if x),
        'topic_keys': sorted(x for x in topic_keys if x),
        'biometric_modalities': sorted(x for x in biometric_modalities if x),
    }

CANDIDATE_KEYS = load_candidate_keys_from_gold()
print({k: len(v) for k, v in CANDIDATE_KEYS.items()})


def retrieve_chunks_by_qnum(qnum: int, top_k: int = SEM_TOP_K):
    question = get_question_text(qnum)
    nodes = retrieve_chunks(question, top_k=top_k)
    chunks = []
    for node in nodes:
        chunks.append({
            'score': getattr(node, 'score', None),
            'metadata': dict(node.metadata or {}),
            'text': node.text,
        })
    return question, chunks


{'cities': 48, 'airports': 18, 'countries': 13, 'doc_types': 17, 'topic_keys': 16, 'biometric_modalities': 1}


In [60]:
# 19) LLM extraction helpers for semantic retrieval

def _responses_json(system_prompt: str, user_payload: dict, schema: dict, model: str = EXTRACT_LLM_MODEL):
    if not GPT_API_KEY:
        raise ValueError('GPT_API_KEY is not set in environment.')
    payload = {
        'model': model,
        'input': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': json.dumps(user_payload, ensure_ascii=False)}
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'semantic_extraction',
                'schema': schema,
                'strict': True,
            }
        }
    }
    resp = requests.post(
        'https://api.openai.com/v1/responses',
        headers={
            'Authorization': f'Bearer {GPT_API_KEY}',
            'Content-Type': 'application/json',
        },
        json=payload,
        timeout=120,
    )
    if not resp.ok:
        raise RuntimeError(f'OpenAI API error {resp.status_code}: {resp.text[:1000]}')
    data = resp.json()
    text = data.get('output_text') or ''
    if not text:
        out = []
        for item in data.get('output', []):
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    out.append(content.get('text', ''))
        text = ''.join(out).strip()
    if not text:
        raise ValueError('Empty JSON output from extraction model')
    return json.loads(text)


def _merge_encounter_fact_rows(rows):
    merged = {}
    for row in rows:
        post_key = row.get('post_key')
        if not post_key:
            continue
        rec = merged.setdefault(post_key, {
            'post_key': post_key,
            'atCountry': set(),
            'atCity': set(),
            'atAirport': set(),
            'document_types': set(),
            'topic_keys': set(),
            'biometric_modalities': set(),
        })
        for field in ['atCountry', 'atCity', 'atAirport', 'document_types', 'topic_keys', 'biometric_modalities']:
            rec[field].update(x for x in (row.get(field) or []) if x)
    return merged


def _fill_country_from_metadata(chunks, merged):
    for chunk in chunks:
        meta = chunk.get('metadata') or {}
        post_id = str(meta.get('post_id') or '').strip()
        if not post_id:
            continue
        post_key = f'post_{post_id}'
        if post_key not in merged:
            continue
        rec = merged[post_key]
        if not rec['atCountry']:
            country_source = meta.get('country_source')
            if country_source in COUNTRY_SOURCE_TO_KEY:
                rec['atCountry'].add(COUNTRY_SOURCE_TO_KEY[country_source])
    return merged


def encounter_rows_to_canonical_triples(encounter_rows):
    triples = set()
    for rec in encounter_rows.values():
        post_key = rec['post_key']
        for x in sorted(rec['atCountry']):
            triples.add(('Encounter', post_key, 'atCountry', 'Country', x))
        for x in sorted(rec['atCity']):
            triples.add(('Encounter', post_key, 'atCity', 'City', x))
        for x in sorted(rec['atAirport']):
            triples.add(('Encounter', post_key, 'atAirport', 'Airport', x))
        for x in sorted(rec['document_types']):
            triples.add(('Encounter', post_key, 'documentType', 'Literal', x))
        for x in sorted(rec['topic_keys']):
            triples.add(('Encounter', post_key, 'topicKey', 'Literal', x))
        for x in sorted(rec['biometric_modalities']):
            triples.add(('Encounter', post_key, 'biometricModality', 'Literal', x))
    return triples


def extract_structured_facts_from_chunks(question: str, chunks, model: str = EXTRACT_LLM_MODEL):
    schema = {
        'type': 'object',
        'additionalProperties': False,
        'required': ['rows'],
        'properties': {
            'rows': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'additionalProperties': False,
                    'required': ['post_key', 'atCountry', 'atCity', 'atAirport', 'document_types', 'topic_keys', 'biometric_modalities'],
                    'properties': {
                        'post_key': {'type': 'string'},
                        'atCountry': {'type': 'array', 'items': {'type': 'string'}},
                        'atCity': {'type': 'array', 'items': {'type': 'string'}},
                        'atAirport': {'type': 'array', 'items': {'type': 'string'}},
                        'document_types': {'type': 'array', 'items': {'type': 'string'}},
                        'topic_keys': {'type': 'array', 'items': {'type': 'string'}},
                        'biometric_modalities': {'type': 'array', 'items': {'type': 'string'}},
                    }
                }
            }
        }
    }

    system = (
        'You extract encounter-level ontology facts from retrieved forum chunks for a semantic RAG baseline. '
        'Return ONLY JSON. Each output object represents one encounter/post. '
        'Set post_key exactly to post_{post_id} from chunk metadata. '
        'Use only facts explicitly supported by the chunk text or by stable chunk metadata. '
        'Use only candidate keys provided by the user. '
        'Allowed scope predicates are atCountry, atCity, atAirport. '
        'Allowed literal fields are document_types, topic_keys, biometric_modalities. '
        'Do not invent graph-internal IDs such as DocumentCheck or DocumentInstance keys. '
        'If a fact is not clear, omit it. '
        'Do not produce prose.'
    )

    payload = {
        'question': question,
        'candidate_keys': CANDIDATE_KEYS,
        'chunks': [
            {
                'post_id': str((c.get('metadata') or {}).get('post_id', '')),
                'country_source': (c.get('metadata') or {}).get('country_source'),
                'source_file': (c.get('metadata') or {}).get('source_file'),
                'text': c.get('text', ''),
            }
            for c in chunks
        ]
    }

    parsed = _responses_json(system, payload, schema, model=model)
    rows = parsed.get('rows', [])
    merged = _merge_encounter_fact_rows(rows)
    merged = _fill_country_from_metadata(chunks, merged)
    return merged


def extract_triples_from_chunks(question: str, chunks, model: str = EXTRACT_LLM_MODEL):
    merged = extract_structured_facts_from_chunks(question, chunks, model=model)
    triples = encounter_rows_to_canonical_triples(merged)
    return triples, merged


In [61]:
# 20) Aggregate computation and graph baseline loading

def _scope_match_encounter(rec, scope):
    if not scope:
        return True
    cities = set(scope.get('cities') or [])
    airports = set(scope.get('airports') or [])
    countries = set(scope.get('countries') or [])
    if cities or airports:
        return bool(cities.intersection(rec['atCity']) or airports.intersection(rec['atAirport']))
    if countries:
        return bool(countries.intersection(rec['atCountry']))
    return True


def _filtered_encounters(encounters, gold_meta):
    scope = (gold_meta or {}).get('scope') or {'cities': [], 'airports': [], 'countries': []}
    return {k: v for k, v in encounters.items() if _scope_match_encounter(v, scope)}


def compute_semantic_aggregate(encounters, gold_meta):
    agg = (gold_meta or {}).get('aggregate') or {}
    if not agg:
        raise ValueError('No aggregate metadata available')

    group_by = agg.get('group_by')
    metric = agg.get('metric')
    order = agg.get('order', 'desc')
    top_k = int(agg.get('top_k', 1))
    doc_types = set((gold_meta or {}).get('doc_types') or [])
    topic_keys = set((gold_meta or {}).get('topic_keys') or [])
    biometric_modalities = set((gold_meta or {}).get('biometric_modalities') or [])

    encounters = _filtered_encounters(encounters, gold_meta)

    if group_by == 'topic':
        counts = {}
        for rec in encounters.values():
            topics = set(rec['topic_keys'])
            if topic_keys:
                topics = topics.intersection(topic_keys)
            for t in topics:
                counts[t] = counts.get(t, 0) + 1
        rows = [{'topic': k, 'metric': v} for k, v in counts.items()]
    else:
        group_field_map = {'country': 'atCountry', 'city': 'atCity', 'airport': 'atAirport'}
        if group_by not in group_field_map:
            raise ValueError(f'Unsupported group_by: {group_by}')
        group_field = group_field_map[group_by]
        counts = {}
        for rec in encounters.values():
            groups = sorted(rec[group_field])
            if not groups:
                continue
            docs = set(rec['document_types'])
            topics = set(rec['topic_keys'])
            bios = set(rec['biometric_modalities'])
            if doc_types:
                docs = docs.intersection(doc_types)
            if topic_keys:
                topics = topics.intersection(topic_keys)
            if biometric_modalities:
                bios = bios.intersection(biometric_modalities)

            for g in groups:
                if metric == 'documents':
                    inc = len(docs)
                elif metric == 'questions':
                    inc = len(topics)
                elif metric == 'biometric':
                    inc = len(bios)
                elif metric == 'encounters':
                    inc = 1
                elif metric == 'no_questions':
                    inc = 1 if len(rec['topic_keys']) == 0 else 0
                elif metric == 'passport_only':
                    inc = 1 if len(rec['topic_keys']) == 0 and docs == {'foreign_passport'} else 0
                else:
                    raise ValueError(f'Unsupported aggregate metric: {metric}')
                if inc:
                    counts[g] = counts.get(g, 0) + inc
        rows = [{group_by: k, 'metric': v} for k, v in counts.items()]

    rows = norm_agg(rows)
    if order == 'asc':
        rows = sorted(rows, key=lambda d: (float(d.get('metric', 0)), list(d.values())[0]))
    else:
        key_field = [k for k in rows[0].keys() if k != 'metric'][0] if rows else group_by
        rows = sorted(rows, key=lambda d: (-float(d.get('metric', 0)), d.get(key_field)))
    if top_k > 0:
        rows = rows[:top_k]
    return rows


def load_graph_metric(qnum: int):
    path = GRAPH_RESULTS_SOURCE
    if not COMPARE_WITH_GRAPH or not path.exists():
        return {
            'graph_available': False,
            'graph_status': 'graph_unavailable',
            'graph_score': None,
            'graph_mismatches': None,
        }
    with path.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if str(row.get('qnum')).strip() == str(qnum):
                score = row.get('score')
                try:
                    score = float(score)
                except Exception:
                    pass
                return {
                    'graph_available': True,
                    'graph_status': row.get('status'),
                    'graph_score': score,
                    'graph_mismatches': row.get('mismatches'),
                }
    return {
        'graph_available': False,
        'graph_status': 'graph_unavailable',
        'graph_score': None,
        'graph_mismatches': None,
    }


In [62]:
# 21) Semantic evaluation and side-by-side comparison

def evaluate_semantic_qnum(qnum: int, top_k: int = SEM_TOP_K, model: str = EXTRACT_LLM_MODEL, print_details: bool = True):
    question, chunks = retrieve_chunks_by_qnum(qnum, top_k=top_k)
    gold_raw, gold_meta, meta_consistent, gold_agg = load_gold(qnum)
    gold_canonical, gold_encounters, _, _ = load_gold_canonical(qnum)

    merged = extract_structured_facts_from_chunks(question, chunks, model=model)
    sem_triples = encounter_rows_to_canonical_triples(merged)

    result = {
        'qnum': qnum,
        'question': question,
        'mode': (gold_meta or {}).get('mode'),
        'semantic_score': None,
        'semantic_status': None,
        'semantic_precision': None,
        'semantic_recall': None,
        'semantic_f1': None,
        'semantic_retrieved_count': len(sem_triples),
        'gold_count': len(gold_canonical),
        'semantic_triples': sem_triples,
        'gold_canonical': gold_canonical,
        'chunks': chunks,
        'merged_facts': merged,
    }

    if (gold_meta or {}).get('mode') == 'aggregate':
        semantic_agg = compute_semantic_aggregate(merged, gold_meta)
        ok = norm_agg(semantic_agg) == norm_agg(gold_agg)
        result['semantic_score'] = 'OK' if ok else 'НЕ ОК'
        result['semantic_status'] = '🟩' if ok else '🟥'
        result['semantic_aggregate'] = semantic_agg
        result['gold_aggregate'] = gold_agg
        if print_details:
            print('qnum', qnum)
            print('question', question)
            print('mode', result['mode'])
            print('semantic aggregate:', semantic_agg)
            print('gold aggregate    :', gold_agg)
            print('semantic status   :', result['semantic_status'])
    else:
        tp = len(sem_triples & gold_canonical)
        precision = tp / len(sem_triples) if sem_triples else 0.0
        recall = tp / len(gold_canonical) if gold_canonical else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        result['semantic_precision'] = precision
        result['semantic_recall'] = recall
        result['semantic_f1'] = f1
        result['semantic_score'] = f1
        result['semantic_status'] = '🟩' if abs(f1 - 1.0) < 1e-12 else '🟥'
        if print_details:
            print('qnum', qnum)
            print('question', question)
            print('mode', result['mode'])
            print('retrieved', len(sem_triples))
            print('gold', len(gold_canonical))
            print('precision', precision)
            print('recall', recall)
            print('f1', f1)
            print('semantic-only extra triples:')
            print(sem_triples - gold_canonical)
    return result


def compare_rags(qnum: int, top_k: int = SEM_TOP_K, model: str = EXTRACT_LLM_MODEL, print_details: bool = True):
    sem = evaluate_semantic_qnum(qnum, top_k=top_k, model=model, print_details=print_details)
    graph = load_graph_metric(qnum)

    sem_score = sem['semantic_score']
    graph_score = graph['graph_score']
    delta = None
    if isinstance(sem_score, float) and isinstance(graph_score, float):
        delta = sem_score - graph_score

    result = {
        'qnum': qnum,
        'question': sem['question'],
        'mode': sem['mode'],
        'semantic_score': sem_score,
        'graph_score': graph_score,
        'delta': delta,
        'semantic_status': sem['semantic_status'],
        'graph_status': graph['graph_status'],
        'graph_available': graph['graph_available'],
    }

    if print_details:
        print('graph baseline:', graph)
        print('comparison row:', result)
        print('')
    return result, sem, graph


def compare_rags_batch(qnums, top_k: int = SEM_TOP_K, model: str = EXTRACT_LLM_MODEL, print_details: bool = True):
    rows = []
    details = {}
    for qnum in qnums:
        row, sem, graph = compare_rags(qnum, top_k=top_k, model=model, print_details=print_details)
        rows.append(row)
        details[qnum] = {'semantic': sem, 'graph': graph}
    df = pd.DataFrame(rows)
    return df, details


In [63]:
# 22) Example: compare one question by qnum
QNUM = 1
comparison_row, semantic_details, graph_details = compare_rags(QNUM, top_k=SEM_TOP_K, model=EXTRACT_LLM_MODEL, print_details=True)
comparison_row


qnum 1
question Which documents do I need to bring to passport control in Paris?
mode single
retrieved 40
gold 299
precision 0.625
recall 0.08361204013377926
f1 0.14749262536873156
semantic-only extra triples:
{('Encounter', 'post_11484564', 'documentType', 'Literal', 'visa'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'hotel_payment_status'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'hotel_payment_card'), ('Encounter', 'post_11484979', 'topicKey', 'Literal', 'visa_schengen'), ('Encounter', 'post_11484979', 'topicKey', 'Literal', 'previous_visits'), ('Encounter', 'post_11484564', 'topicKey', 'Literal', 'visa_schengen'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'employment'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'visit_purpose'), ('Encounter', 'post_11769523', 'topicKey', 'Literal', 'visit_duration'), ('Encounter', 'post_11769523', 'topicKey', 'Literal', 'accommodation_details'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'paym

{'qnum': 1,
 'question': 'Which documents do I need to bring to passport control in Paris?',
 'mode': 'single',
 'semantic_score': 0.14749262536873156,
 'graph_score': 1.0,
 'delta': -0.8525073746312685,
 'semantic_status': '🟥',
 'graph_status': '🟩',
 'graph_available': True}

In [64]:
# 23) Example: batch comparison for a list of qnum
QNUMS = [1, 9, 10]
comparison_df, comparison_details = compare_rags_batch(QNUMS, top_k=SEM_TOP_K, model=EXTRACT_LLM_MODEL, print_details=True)
comparison_df


qnum 1
question Which documents do I need to bring to passport control in Paris?
mode single
retrieved 36
gold 299
precision 0.6944444444444444
recall 0.08361204013377926
f1 0.1492537313432836
semantic-only extra triples:
{('Encounter', 'post_11484564', 'documentType', 'Literal', 'visa'), ('Encounter', 'post_11870493', 'topicKey', 'Literal', 'visa_schengen'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'hotel_payment_status'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'hotel_payment_card'), ('Encounter', 'post_11484564', 'topicKey', 'Literal', 'visa_schengen'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'employment'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'visit_purpose'), ('Encounter', 'post_11769523', 'topicKey', 'Literal', 'visit_duration'), ('Encounter', 'post_11769523', 'topicKey', 'Literal', 'accommodation_details'), ('Encounter', 'post_3902451', 'topicKey', 'Literal', 'cash_amount'), ('Encounter', 'post_3902451', 'topicKey', 'Literal'

,qnum,question,mode,semantic_score,graph_score,delta,semantic_status,graph_status,graph_available
0,1,Which documents do I need to bring to passport...,single,0.149254,1.0,-0.850746,🟥,🟩,True
1,9,Which documents are checked for travelers who ...,conditional,0.392857,1.0,-0.607143,🟥,🟩,True
2,10,Which question topics are asked of travelers w...,conditional,0.121951,1.0,-0.878049,🟥,🟩,True


In [65]:
# 24) Optional: inspect semantic extracted canonical triples for one qnum
QNUM = 1
_, sem_details, _ = compare_rags(QNUM, top_k=SEM_TOP_K, model=EXTRACT_LLM_MODEL, print_details=False)
print('QUESTION:', sem_details['question'])
print('MODE:', sem_details['mode'])
print('SEMANTIC TRIPLES:')
for t in sorted(sem_details['semantic_triples']):
    print(t)


QUESTION: Which documents do I need to bring to passport control in Paris?
MODE: single
SEMANTIC TRIPLES:
('Encounter', 'post_11484564', 'atAirport', 'Airport', 'airport_cdg')
('Encounter', 'post_11484564', 'atCity', 'City', 'city_paris')
('Encounter', 'post_11484564', 'atCountry', 'Country', 'country_france')
('Encounter', 'post_11484564', 'documentType', 'Literal', 'visa')
('Encounter', 'post_11484564', 'topicKey', 'Literal', 'visa_schengen')
('Encounter', 'post_11484979', 'atAirport', 'Airport', 'airport_cdg')
('Encounter', 'post_11484979', 'atCity', 'City', 'city_paris')
('Encounter', 'post_11484979', 'atCountry', 'Country', 'country_france')
('Encounter', 'post_11484979', 'documentType', 'Literal', 'visa')
('Encounter', 'post_11484979', 'topicKey', 'Literal', 'previous_visits')
('Encounter', 'post_11769523', 'atAirport', 'Airport', 'airport_cdg')
('Encounter', 'post_11769523', 'atCity', 'City', 'city_paris')
('Encounter', 'post_11769523', 'atCountry', 'Country', 'country_france')


In [66]:
# 25) Optional: inspect semantic aggregate result for aggregate questions
QNUM = 3
_, sem_details, _ = compare_rags(QNUM, top_k=SEM_TOP_K, model=EXTRACT_LLM_MODEL, print_details=False)
print('QUESTION:', sem_details['question'])
print('MODE:', sem_details['mode'])
print('SEMANTIC AGGREGATE:', sem_details.get('semantic_aggregate'))
print('GOLD AGGREGATE    :', sem_details.get('gold_aggregate'))


QUESTION: Which question topics are most common at passport control in France?
MODE: aggregate
SEMANTIC AGGREGATE: [{'metric': 2, 'topic': 'payment_card'}, {'metric': 2, 'topic': 'visit_purpose'}, {'metric': 1, 'topic': 'accommodation_details'}, {'metric': 1, 'topic': 'cash_amount'}, {'metric': 1, 'topic': 'exit_date_from_schengen'}, {'metric': 1, 'topic': 'visa_schengen'}, {'metric': 1, 'topic': 'visit_duration'}]
GOLD AGGREGATE    : [{'topic': 'visit_duration', 'metric': 11}, {'topic': 'visit_purpose', 'metric': 11}, {'topic': 'accommodation_details', 'metric': 8}, {'topic': 'itinerary_transport', 'metric': 4}, {'topic': 'cash_amount', 'metric': 4}, {'topic': 'previous_visits', 'metric': 3}, {'topic': 'travel_companions', 'metric': 2}, {'topic': 'exit_date_schengen', 'metric': 2}, {'topic': 'exit_date_from_schengen', 'metric': 2}, {'topic': 'employment', 'metric': 2}]
